# Citation Index API — interactive client test

Run this notebook from the project root with the package installed:

```bash
pip install -e ".[dev]"
pip install ipykernel   # if needed
python -m ipykernel install --user --name citation-index-api-client
```

Select that kernel in Jupyter / VS Code, then run the cells top to bottom.

In [1]:
from pathlib import Path

from citation_index_api_client import CitationIndexClient
from citation_index_api_client.exceptions import APIError, JobFailedError, JobTimeoutError

# --- configure ---
BASE_URL = "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl"
PDF_PATH = Path("paper.pdf")  # set to your PDF, or None to skip text extraction
POLL_INTERVAL = 2.0
JOB_TIMEOUT = 600.0

client = CitationIndexClient(base_url=BASE_URL, timeout=120.0)
print(f"Client → {client.base_url}")

Client → https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl


## Health & API info

In [2]:
try:
    print("health:", client.health())
    print("root:", client.root())
except APIError as e:
    print(f"API error {e.status_code}: {e.detail}")

health: {'status': 'healthy', 'redis': 'ok', 'storage': 'ok', 'version': '0.1.0'}
root: {'name': 'Citation Index API', 'version': '0.1.0', 'docs': '/docs', 'health': '/health'}


## Extract text from a PDF

Set `PDF_PATH` in the first cell to a local `.pdf` file. This enqueues an async job and waits for the result.

In [ ]:
PDF_PATH = Path("../test_data/reg_0035-2039_1989_num_102_485_2445.pdf")

In [6]:
if PDF_PATH is None or not PDF_PATH.exists():
    print(f"Skipping text extraction — set PDF_PATH to an existing file (got {PDF_PATH!r})")
else:
    job = client.extract_text(PDF_PATH, extractor="pymupdf", markdown=True)
    print(f"Enqueued: {job}")

    status = client.wait_for_job(
        job.job_id,
        poll_interval=POLL_INTERVAL,
        timeout=JOB_TIMEOUT,
    )
    print(f"Finished: {status}")

    text_result = client.get_job_result(job.job_id)
    if isinstance(text_result, dict):
        preview = str(text_result)[:500]
    else:
        preview = str(text_result)[:500]
    print(f"Result preview:\n{preview}...")

Enqueued: JobResponse(job_id='7feae48f-08fd-4ba5-8bf0-254ef8520dcf', status='queued', created_at='2026-05-29T09:36:38.786887', message='Text extraction job enqueued')
Finished: JobStatus(job_id='7feae48f-08fd-4ba5-8bf0-254ef8520dcf', status='completed', created_at='2026-05-29T09:36:38.786887', started_at='2026-05-29T09:36:38.877828', completed_at='2026-05-29T09:36:44.822088', current_stage='text_extraction', completed_stages=['text_extraction'], error=None, progress=None)
Result preview:
{'text': "### Revue des Études Grecques\n\n# Sur le troisième stasimon d'Œdipe-Roi\n\n### Albert Machin\n\nCiter ce document / Cite this document :\n\n\nMachin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvier\njuin 1989. pp. 192-201 ;\n\n\n[doi : https://doi.org/10.3406/reg.1989.2445](https://doi.org/10.3406/reg.1989.2445)\n\n\n[https://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445](https://www.persee.fr/doc/reg_0035-2039_...


In [10]:
text_result

{'text': "### Revue des Études Grecques\n\n# Sur le troisième stasimon d'Œdipe-Roi\n\n### Albert Machin\n\nCiter ce document / Cite this document :\n\n\nMachin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvier\njuin 1989. pp. 192-201 ;\n\n\n[doi : https://doi.org/10.3406/reg.1989.2445](https://doi.org/10.3406/reg.1989.2445)\n\n\n[https://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445](https://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445)\n\n\nFichier pdf généré le 18/04/2018\n\n\n## SUR LE TROISIEME STASIMON O'ŒDIPE-ROI\n\n{O.R. vientLe de 1086-1 chant proférer109),du sur en Chœur réponse sa naissance dans aux lequel affirma(il seraiest **t** ions fils évoquée de pleines «Fortune l'origide cola **n** fiance e Généredivine q **u** se»,e d'Œdipe le héros1080\n\n\n81) se situe en un moment capital : entre la scène du Messager de Corinthe, qui\nvient de révéler qu 'Œdipe n'est pas le fils de Polybe et de Mérop

## Extract references from markdown

Uses a small sample bibliography block. Replace `SAMPLE_MARKDOWN` with output from text extraction if you have it.

In [ ]:
job = client.extract_references(
    text_result['text'],
    method="full_text",
    temperature=0.3,
)
print(f"Enqueued: {job}")

try:
    refs_result = client.wait_for_result(
        job.job_id,
        poll_interval=POLL_INTERVAL,
        timeout=JOB_TIMEOUT,
    )
    refs_result
except JobFailedError as e:
    print(f"Job failed: {e}")
except JobTimeoutError as e:
    print(f"Timed out after {e.timeout}s")

Enqueued: JobResponse(job_id='3533b94f-7a31-4772-95cc-1211c61341b5', status='queued', created_at='2026-05-29T09:42:41.419359', message='Reference extraction job enqueued')


In [25]:
refs_result

{'references': ["Machin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvier juin 1989. pp. 192-201 ;",
  'C. M. Bowra, Sophoclean Tragedy, Oxford, 1944, p. 199.',
  'A. Lesky, Greek Tragedy, éd. angl. Londres, 1965, p. 113.',
  'R. C. Jebb éd. O.T. Cambridge, 1914, p. 143. J. C. Kamerbeek éd. O.T. Leyde, 1967, p. 208.',
  "R. W. B. Burton, The Chorus in Sophocles' Tragedies, Oxford, 1980, p. 169-70. H. D. F. Kitto, Greek Tragedy, 3e éd., Londres, 1970, p. 167 : «lyrical instrument».",
  'V. en part. R. C. Jebb, op. cit., comm. à 1086-1109. W. Kranz, Stasimon, Berlin, 1933, p. 213. G. M. Kirkwood, A Study of Sophoclean Drama, New York, 1958, p. 199-200. J. C. Kamerbeek, op. cit., comm. à 1086-1109.',
  'T. B. L. Webster, An Introduction to Sophocles, Oxford, 1936, p. 105.',
  'Jebb; éd. Trachiniennes p. lxi, Antig. p. lxix, Ajax p. lxviii. M. A. Bayfield, éd. Antigone, N-Y, 1968, p. xxm. G. Mùller, éd. Antigone, Leyde, 1967,

## Parse reference strings

Send raw bibliography lines to the parser endpoint.

In [27]:
job = client.parse_references(refs_result['references'], parser="grobid", temperature=0.0)
print(f"Enqueued: {job}")

parsed_result = client.wait_for_result(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
parsed_result

Enqueued: JobResponse(job_id='befedb79-c450-41b3-b704-a44fdb395c4c', status='queued', created_at='2026-05-29T09:47:54.735571', message='Reference parsing job enqueued')


{'references': [{'full_title': "Sur le troisième stasimon d'Œdipe-Roi",
   'journal_title': 'Revue des Études Grecques',
   'authors': [{'first_name': 'Machin',
     'middle_name': None,
     'surname': 'Albert',
     'name_link': None,
     'role_name': None}],
   'editors': None,
   'publisher': None,
   'translator': None,
   'publication_place': None,
   'publication_year': 1989,
   'publication_date_raw': 'Janvier juin 1989',
   'identifiers': [],
   'ref_type': None,
   'raw': {'tei': {'analytic_title': "Sur le troisième stasimon d'Œdipe-Roi"}},
   'volume': '102',
   'issue': None,
   'pages': None,
   'cited_range': None,
   'footnote_number': None},
  {'full_title': 'Sophoclean Tragedy',
   'journal_title': None,
   'authors': [{'first_name': 'C',
     'middle_name': 'M',
     'surname': 'Bowra',
     'name_link': None,
     'role_name': None}],
   'editors': None,
   'publisher': None,
   'translator': None,
   'publication_place': 'Oxford',
   'publication_year': 1944,
   'p

In [29]:
len(parsed_result['references'])

37

## Manual job polling (optional)

Use this pattern when you want to inspect status between polls instead of blocking on `wait_for_result`.

In [30]:
# Re-use the last job_id from the cell above, or paste one here:
job_id = job.job_id

status = client.get_job_status(job_id)
print(status)
print("terminal?", status.is_terminal)

if status.status == "completed":
    result = client.get_job_result(job_id)
    print(result)

JobStatus(job_id='befedb79-c450-41b3-b704-a44fdb395c4c', status='completed', created_at='2026-05-29T09:47:54.735571', started_at=None, completed_at='2026-05-29T09:47:57.841824', current_stage='reference_parsing', completed_stages=['reference_parsing'], error=None, progress=None)
terminal? True
{'references': [{'full_title': "Sur le troisième stasimon d'Œdipe-Roi", 'journal_title': 'Revue des Études Grecques', 'authors': [{'first_name': 'Machin', 'middle_name': None, 'surname': 'Albert', 'name_link': None, 'role_name': None}], 'editors': None, 'publisher': None, 'translator': None, 'publication_place': None, 'publication_year': 1989, 'publication_date_raw': 'Janvier juin 1989', 'identifiers': [], 'ref_type': None, 'raw': {'tei': {'analytic_title': "Sur le troisième stasimon d'Œdipe-Roi"}}, 'volume': '102', 'issue': None, 'pages': None, 'cited_range': None, 'footnote_number': None}, {'full_title': 'Sophoclean Tragedy', 'journal_title': None, 'authors': [{'first_name': 'C', 'middle_name':

In [31]:
client.close()
print("Client closed.")

Client closed.
